In [1]:
import re
from tqdm import tqdm #Progress Indication
from pprint import pprint #Saubere JSON darstellung

import numpy as np 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import defaultdict
from collections import Counter

import spacy # Lemmatizer

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

#import matplotlib.pyplot as plt
#import matplotlib.ticker as mtick

**Beschriebenes Vorgehen:**

Bereinigung durch:
- Pandas
- NLTK
- SpaCy (besser für Deutsches Lemmatizing)

-> Separat für Topic Modeling/ Semantik Analyse


Vektorisierung anhand BoW und TF-IDF + n-Gramme durch:
- sklearn

-> Darstellung Unterschied zwischen BoW u TF-IDF
-> TF-IDF sprachsepariert, da sonst falsche stoppwörter


Themenidentifikation (LSA/LDA) durch:
- sklearn
- vaderSentiment
- GerVADER

-> Ausarbeitung bester Ansatz:

BoW + LDA
TF-IDF + LDA
-> TF-IDF + LSA



In [2]:
def main():

    # To be filled   
    print("To be filled")

if __name__ == "__main__":
    main()

To be filled


In [28]:
#### Analyse Funktionen #######
def style_compare_table(df, caption=None):
    count_cols = [
        ("de", "Topic", "Count"),
        ("de", "Sentiment", "Count"),
        ("en", "Topic", "Count"),
        ("en", "Sentiment", "Count"),
    ]

    styled = df.style

    if caption:
        styled = styled.set_caption(caption)

    styled = (
        styled
        .hide(axis="index")
        .set_table_styles([
            # dünn: Topic | Sentiment innerhalb de
            {
                "selector": "th.col1, td.col1",
                "props": [("border-right", "1px solid black")]
            },
            # dick: de ||| en
            {
                "selector": "th.col3, td.col3",
                "props": [("border-right", "4px solid black")]
            },
            # dünn: Topic | Sentiment innerhalb en
            {
                "selector": "th.col5, td.col5",
                "props": [("border-right", "1px solid black")]
            },
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold")
                ]
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("font-size", "16px")
                ]
            }
        ])
        .set_properties(**{"text-align": "left"})
        .set_properties(
            subset=count_cols,
            **{"text-align": "right"}
        )
        .background_gradient(
            subset=count_cols,
            cmap="Blues"
        )
    )

    return styled

def analyze_tokens(data_by_lang, top_n=30):
    data = {}

    for lang in ["de", "en"]:
        sentences = data_by_lang[lang]["sentences"]
        tokens = [word for sent in sentences for word in sent]
        counts = Counter(tokens).most_common(top_n)

        data[(lang, "Token")] = [w for w, _ in counts]
        data[(lang, "Count")] = [c for _, c in counts]

    df = pd.DataFrame(data)
    df.columns = pd.MultiIndex.from_tuples(df.columns)

    return style_token_table(
        df,
        split_after_col=1,
        count_cols=[("de", "Count"), ("en", "Count")],
        caption=f"Top {top_n} Tokens (de vs en)"
    )

def compare_top_tokens(data_topic, data_sentiment, top_n=30):
    """
    Vergleich Topic vs Sentiment für DE und EN gleichzeitig
    """

    def get_counts(data, lang):
        tokens = [
            word
            for sent in data[lang]["sentences"]
            for word in sent
        ]
        return Counter(tokens).most_common(top_n)

    # Counts holen
    topic_de = get_counts(data_topic, "de")
    sent_de  = get_counts(data_sentiment, "de")

    topic_en = get_counts(data_topic, "en")
    sent_en  = get_counts(data_sentiment, "en")

    # DataFrame bauen
    df = pd.DataFrame({
        ("de", "Topic", "Token"): [w for w, _ in topic_de],
        ("de", "Topic", "Count"): [c for _, c in topic_de],
        ("de", "Sentiment", "Token"): [w for w, _ in sent_de],
        ("de", "Sentiment", "Count"): [c for _, c in sent_de],

        ("en", "Topic", "Token"): [w for w, _ in topic_en],
        ("en", "Topic", "Count"): [c for _, c in topic_en],
        ("en", "Sentiment", "Token"): [w for w, _ in sent_en],
        ("en", "Sentiment", "Count"): [c for _, c in sent_en],
    })

    df.columns = pd.MultiIndex.from_tuples(df.columns)

    return style_compare_table(
        df,
        caption=f"Top {top_n} Tokens: Topic vs Sentiment (de/en)"
    )
    
def print_lda_topics(model, feature_names, n_top_words=10):
    """
    Gibt die wichtigsten Wörter je Topic aus.
    """
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]

        print(f"\nTopic {topic_idx + 1}:")
        print(", ".join(top_words))

def topics_matrix(model, feature_names, n_top_words=10, prefix="Topic"):
    """
    Erstellt eine Topic-Tabelle für ein einzelnes Modell.
    Spalten: Topic1, Topic2, ...
    Zeilen: Top-Wörter
    """
    topics = {}

    for i, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-n_top_words:][::-1]
        topics[f"{prefix}{i+1}"] = [feature_names[j] for j in top_idx]

    return pd.DataFrame(topics)


def compare_topic_models_multiindex(models, n_top_words=10):
    first_model = next(iter(models.values()))["model"]
    n_topics = first_model.components_.shape[0]

    data = {}

    for topic_idx in range(n_topics):
        for method_name, d in models.items():
            topic = d["model"].components_[topic_idx]
            features = d["features"]

            top_idx = topic.argsort()[-n_top_words:][::-1]
            words = [features[i] for i in top_idx]

            data[(f"Topic {topic_idx+1}", method_name)] = words

    return pd.DataFrame(data)


In [24]:
def batch_lemmatizer(texts, nlp, stopw, desc, n_process):
    sentences = []

    reviews = nlp.pipe(texts, batch_size=200, n_process=n_process)

    for review in tqdm(reviews, total=len(texts), desc=desc):
        lemmas = [
            token.lemma_.lower()
            for token in review
            if token.is_alpha
            and len(token) > 2
            and not token.is_stop
            and token.lemma_.lower() not in stopw
        ]

        if lemmas:
            sentences.append(lemmas)

    return sentences
            
def clean_and_tokenize(df, stopw_de, stopw_en, n_process):
    stopw_de = set(stopw_de)
    stopw_en = set(stopw_en)

    nlp_de = spacy.load("de_core_news_sm", disable=["parser", "ner"])
    nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    df_de = df[df["language"] == "de"]["review"].fillna("").astype(str)
    df_en = df[df["language"] == "en"]["review"].fillna("").astype(str)

    result = {}

    for lang, texts, nlp, stopw, desc in [
        ("de", df_de, nlp_de, stopw_de, "Deutsch verarbeiten"),
        ("en", df_en, nlp_en, stopw_en, "Englisch verarbeiten"),
    ]:
        sentences = batch_lemmatizer(
            texts=texts,
            nlp=nlp,
            stopw=stopw,
            desc=desc,
            n_process=n_process
        )

        documents = [" ".join(sentence) for sentence in sentences]
        vocabulary = sorted(set(word for sent in sentences for word in sent))
        index = {word: i for i, word in enumerate(vocabulary)}

        result[lang] = {
            "sentences": sentences,
            "documents": documents,
            "vocabulary": vocabulary,
            "index": index
        }

    return result

def vectorize(Vectorizer, data_by_lang):
    params = {
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.8,
        "dtype": np.float32,
    }

    results = {}

    for lang, data in data_by_lang.items():
        documents = data["documents"]

        vectorizer = Vectorizer(**params)
        matrix = vectorizer.fit_transform(documents)

        results[lang] = {
            "matrix": matrix,
            "vectorizer": vectorizer,
            "feature_names": vectorizer.get_feature_names_out(),
            "documents": documents,
            "sentences": data["sentences"],
            "vocabulary": data["vocabulary"],
            "index": data["index"]
        }

        print(f"{lang}: {matrix.shape[0]} Dokumente, {matrix.shape[1]} Features")

    return results
    

In [5]:
##########
# Variable Definition
##########
local_dir = "data"
path_reviews = f"{local_dir}/combined_reviews.csv"

# Manuelle Stopwörter (werden iterativ ergänzt) 
stopw_topic_manual_de, stopw_sentiment_manual_de = [],[]
stopw_topic_manual_en, stopw_sentiment_manual_en = [],[]

# Stoppwörter aus NTLK
stopw_de = stopwords.words("german")
stopw_en = stopwords.words("english") 

# Zusammengesetzte Stopwörter aus manuell und NTLK
stopw_topic_de = stopw_de + stopw_topic_manual_de
stopw_topic_en = stopw_en + stopw_topic_manual_en
stopw_sentiment_de = stopw_de + stopw_sentiment_manual_de
stopw_sentiment_en = stopw_en + stopw_sentiment_manual_en

############
# Import data to pandas dataframe
############
df_reviews = pd.read_csv(path_reviews)

df_reviews["language"] = df_reviews["origin"].map({
    "FragdenStaat": "de",
    "YELP": "en",
})

############
# Prepare Data
############

# --- Stopwort-Definitionen ---
stopword_updates = {
    "iter_1": {
        "de_common": [
            "august","information","art","aktuell","zahl","genannt","frage","antwort","fall",
            "stelle","rahmen","bitten","insbesondere","erachten","tatsächlich",
            "bayerisch","bayern","baydsg","bayuig","vig", # Verwaltungsbegriffe
            "lebensmittelbetriebe","routinekontrolle","rüb","avv","lfgb" # Standardanfragen
        ],
        "en_common": [
            "want","look","know","think","people","day","find","tell","ask","take","work"
        ],
        "en_topic_only": [
            "nice","well","delicious","friendly","definitely"
        ]
    },

    "iter_2": {
        "de_common": [
            "bitte","einschließlich","datum","folgend","liegen","antrag","behörde","anfrage",
            "projekt","dokument","erfolgen","zuständig","zuständigkeitsbereich","falls",
            "auskunft","entsprechend","befinden","angabe","betreffen",
            "elektronisch","öffentlich","soweit","überprüfen","registriert", # Juristisch
            "aktenauskunft","gesetz","umweltinformationsgesetz" # Juristisch
        ],
        "en_common": [
            "place","food","get","try","time","come","go",
            "need","way","say","staff","experience"
        ],
        "en_topic_only": [
            "good","great","like","love","little"
        ]
    },

    "iter_3": { 
        "de_common": [ # Hauptsächlich Verwaltungsfloskeln 
            "letzter","form","begründung","interesse",
            "sämtlicher","anzahl","gemeinde","handeln",
            "geplant","sinn","monat", "freundlich",
            "senden","mitteilen","grüße","häufig"
            "stellen","höhe","angeben"
        ],
        "en_common": [ 
            "lot","feel","thing","visit", "pretty",
            "long","area","minute","new"
        ],
        "en_topic_only": ["bad","amazing"]
    },
    
    "iter_4": { 
        "de_common": [ 
            "unverzüglich", "ausdrücklich", "häufig", 
            "vorab", "bewerten", "übersicht", "jährlich", "mühe",
            "danken", "verweisen", "zugänglich"
        ],
        "en_common": [ 
            "sure", "right", "review", "hour","leave"
        ],
        "en_topic_only": ["recommend", "enjoy"]
    },
    
    "iter_5": { # Nach BoW + LDA Topic Modeling
        "de_common": [ 
            "satz", "empfangsbestätigung", "widersprechen", "weiterzuleiten",
            "unterrichten", "weitergabe", "aufwand", "gebührenpflichtig",
            "herr", "geehrt", "dame", "gemäß", "beantragen","gmbh",
            "vorhanden", "vorliegen", "zugang", "angefragt",
            "gewähren", "fragdenstaat", "verfahren", "gesetzlich",
            "grund", "bezug", "mitteilung"
        ],
        "en_common": [],
        "en_topic_only": []
    },

    "iter_6": { # Nach 2tem BoW + LDA Topic Modeling Durchlauf
        "de_common": [ 
            "einfach", "somit", "spätestens", "stellen",
            "zusätzlich", "sofern", "aufgrund"
        ],
        "en_common": [],
        "en_topic_only": []
    }
}

for iteration in stopword_updates.values():

    # Deutsch (immer beide)
    stopw_topic_de += iteration["de_common"]
    stopw_sentiment_de += iteration["de_common"]

    # Englisch (gemeinsam)
    stopw_topic_en += iteration["en_common"]
    stopw_sentiment_en += iteration["en_common"]

    # Englisch (nur Topic)
    stopw_topic_en += iteration["en_topic_only"]

# ggf. Duplikate entfernen
stopw_topic_de = sorted(set(stopw_topic_de))
stopw_sentiment_de = sorted(set(stopw_sentiment_de))
stopw_topic_en = sorted(set(stopw_topic_en))
stopw_sentiment_en = sorted(set(stopw_sentiment_en))


# Daten für Topic Modeling
print("Cleaning Data - Topic Modeling")
data_topic = clean_and_tokenize(
    df_reviews,
    stopw_de=stopw_topic_de,
    stopw_en=stopw_topic_en,
    n_process=2 # ggf. Anpassen für schnellere Verarbeitung
)

# Daten für Sentimentanalyse
print("\nCleaning Data - Sentiment Analysis")
data_sentiment = clean_and_tokenize(
    df_reviews,
    stopw_de=stopw_sentiment_de,
    stopw_en=stopw_sentiment_en,
    n_process=2
)


Cleaning Data - Topic Modeling


Englisch verarbeiten: 100%|████████████████████████████████████████████████████████| 1000/1000 [00:12<00:00, 77.25it/s]



Cleaning Data - Sentiment Analysis


Englisch verarbeiten: 100%|████████████████████████████████████████████████████████| 1000/1000 [00:14<00:00, 68.59it/s]


In [29]:
compare_top_tokens(data_topic, data_sentiment, top_n=15)

In [31]:
############
# Create Vectors with BoW & TF-IDF
# Separate Pipelines: Topic Modeling vs. Sentiment
############

print("Creating Vectors - Topic Modeling")
vectors_topic = {}
vectors_topic["bow_by_lang"] = vectorize(CountVectorizer, data_topic)
vectors_topic["tfidf_by_lang"] = vectorize(TfidfVectorizer, data_topic)

print("\nCreating Vectors - Sentiment Analysis")
vectors_sentiment = {}
vectors_sentiment["bow_by_lang"] = vectorize(CountVectorizer, data_sentiment)
vectors_sentiment["tfidf_by_lang"] = vectorize(TfidfVectorizer, data_sentiment)

Creating Vectors - Topic Modeling
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2416 Features
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2416 Features

Creating Vectors - Sentiment Analysis
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2641 Features
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2641 Features


In [32]:
############
# Topic Modeling
############

n_topics = 10
n_top_words = 15
max_iter = 20

# --- BoW + LDA sprachgetrennt ---
lda_bow_topics_by_lang = {}

for lang, data in vectors_topic["bow_by_lang"].items():
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="online",
        max_iter=1,
        evaluate_every=-1
    )

    for _ in tqdm(range(max_iter), desc=f"LDA BoW Topic trainieren ({lang})"):
        lda_model.partial_fit(data["matrix"])

    lda_bow_topics_by_lang[lang] = {
        "model": lda_model,
        "matrix": data["matrix"],
        "features": data["feature_names"],
        "documents": data["documents"]
    }

# --- TF-IDF + LDA sprachgetrennt ---

lda_tfidf_topics_by_lang = {}

for lang, data in vectors_topic["tfidf_by_lang"].items():
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="online",
        max_iter=1,
        evaluate_every=-1
    )

    for _ in tqdm(range(max_iter), desc=f"LDA TF-IDF Topic trainieren ({lang})"):
        lsa_matrix = lda_model.partial_fit(data["matrix"])

    lda_tfidf_topics_by_lang[lang] = {
        "model": lda_model,
        "matrix": data["matrix"],
        "features": data["feature_names"]
    }

# --- TF-IDF + LSA sprachgetrennt ---
lsa_tfidf_topics_by_lang = {}

for lang, data in tqdm(vectors_topic["tfidf_by_lang"].items(), desc="LSA TF-IDF Topic trainieren (de/en)"):

    lsa_model = TruncatedSVD(
        n_components=n_topics,
        random_state=42
    )

    lsa_matrix = lsa_model.fit_transform(data["matrix"])

    lsa_tfidf_topics_by_lang[lang] = {
        "model": lsa_model,
        "matrix": lsa_matrix,
        "features": data["feature_names"]
    }

LSA TF-IDF Topic trainieren (de/en): 100%|███████████████████████████████████████████████| 2/2 [00:00<00:00, 27.10it/s]


In [90]:
df_bow_lda_en = topics_matrix(
    lda_bow_topics_by_lang["en"]["model"],
    vectors_topic["bow_by_lang"]["en"]["feature_names"],
    n_top_words=10
)

df_tfidf_lda_en = topics_matrix(
    lda_tfidf_topics_by_lang["en"]["model"],
    vectors_topic["tfidf_by_lang"]["en"]["feature_names"],
    n_top_words=10
)

df_tfidf_lsa_en = topics_matrix(
    lsa_tfidf_topics_by_lang["en"]["model"],
    vectors_topic["tfidf_by_lang"]["en"]["feature_names"],
    n_top_words=10
)


df_bow_lda_de = topics_matrix(
    lda_bow_topics_by_lang["de"]["model"],
    vectors_topic["bow_by_lang"]["de"]["feature_names"],
    n_top_words=10
)

df_tfidf_lda_de = topics_matrix(
    lda_tfidf_topics_by_lang["de"]["model"],
    vectors_topic["tfidf_by_lang"]["de"]["feature_names"],
    n_top_words=10
)

df_tfidf_lsa_de = topics_matrix(
    lsa_tfidf_topics_by_lang["de"]["model"],
    vectors_topic["tfidf_by_lang"]["de"]["feature_names"],
    n_top_words=10
)


from IPython.display import display, Markdown

# -------- Englisch --------
display(Markdown("## BoW + LDA (EN)"))
display(df_bow_lda_en)

display(Markdown("## TF-IDF + LDA (EN)"))
display(df_tfidf_lda_en)

display(Markdown("## TF-IDF + LSA (EN)"))
display(df_tfidf_lsa_en)


# -------- Deutsch --------
display(Markdown("## BoW + LDA (DE)"))
display(df_bow_lda_de)

display(Markdown("## TF-IDF + LDA (DE)"))
display(df_tfidf_lda_de)

display(Markdown("## TF-IDF + LSA (DE)"))
display(df_tfidf_lsa_de)

## BoW + LDA (EN)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,pizza,kitchen,room,fresh,clean,order,lunch,price,order,coffee
1,hair,nail,hotel,salad,stay,wait,chicken,service,table,car
2,use,open,check,order,large,taco,drink,store,friend,customer
3,absolutely,service,awesome,bread,door,burger,beer,sushi,night,service
4,beef,owner,seating,menu,flavor,drink,bar,roll,bar,wait
5,close,fan,treat,service,type,bbq,hot,month,restaurant,call
6,serve,later,wedding,meat,restaurant,bean,sandwich,reasonable,sit,pay
7,flavor,onion,guest,eat,room,atmosphere,salad,buy,cake,line
8,burger,drive,burger,wine,italian,tip,tasty,selection,service,bring
9,pork,finally,happy,chicken,star,eat,fry,sandwich,server,let


## TF-IDF + LDA (EN)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,pizza,wine,burger,salad,room,pizza,roll,chicken,cake,wait
1,order,awesome,sushi,order,store,fresh,sushi,ice,change,bar
2,hair,service,donut,taco,stay,fry,tea,ice cream,price,service
3,table,wedding,frozen,service,car,best,soup,cream,service,burger
4,drink,beer,kid,chicken,hotel,burger,lunch,coffee,fair,menu
5,wait,wonderful,treat,sauce,helpful,line,chinese,reasonable,server,car
6,service,bartender,chicken,bread,clean,flavor,oyster,sandwich,rude,coffee
7,restaurant,music,pet,meat,nail,breakfast,cafe,cute,indian,call
8,awesome,beautiful,eat,eat,nashville,perfect,order,chocolate,relaxed,drink
9,vegetarian,close,excellent,meal,service,wall,pho,awesome,attentive,stop


## TF-IDF + LSA (EN)

...

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,verfahrensstand,unterlage,verbraucherinformation,betrieb,maßnahme,tier,münchen,arzt,waffenschein,fördermeng
1,baubeginn,schule,kosten,bußgeld,vollständig,behördlich,straße,medizinisch,verfassungsfeindlich,unternehmen
2,bebauungsplanverfahr,ministerium,umweltinformation,werbung,biber,bundesland,stadt,meldung,afd,genehmigt
3,gemeindegebiet,thema,gebühr,plattform,konkret,aufschlüsseln,person,unerwünscht,vereinigung,genehmigt fördermeng
4,gemarkung,kommunikation,einschlägig,unzulässig,gebiet,dokumentiert,finden,verdacht,verfassungsfeindlich vereinigung,kommunal
5,aktueller,politisch,fallen,verstoß,staatsministerium,haus,stehen,bevölkerung,person,wasser
6,bebauungsplan,vertrag,geringfügig,erfolgt,regierung,überwachung,derzeit,unterweisung,mitglied,genehmigung
7,aktueller verfahrensstand,aktivität,datenschutzgesetz,herstellen,zeitraum,schlachthof,frau,deutsch,waffenbesitzkarte,menge
8,satzungsbeschluß,nutzen,ablauf,berücksichtigung,artikel,differenzieren,laut,hintergrund,waffenbesitzkarte waffenschein,entrichten
9,verfolgen,klimaneutralität,erbeten,antworten,genehmigung,betrieb,maßnahme,verfügung,alternative,kalenderjahr


## TF-IDF + LDA (DE)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,aufgabe,gutachten,verbraucherinformation,bußgeld,fördermeng,münchen,nürnberg,dokumentiert geschlachtet,verfahrensstand,gymnasium
1,erwartungshorizonte,klimaneutralität,kosten,betrieb,genehmigt,polizei,jva,behördlich überwachung,bebauungsplanverfahr,steuer
2,fach,aktivität,einschlägig,werbung,genehmigt fördermeng,straße,frau,berichtsjahr,baubeginn,weisung
3,aufgabe erwartungshorizonte,lehrkraft,kosten geringfügig,werbung social,kommunal,unterlage,umgang,tierart,gemeindegebiet,bay
4,erwartungshorizonte lösung,ministerium,bürgeranfrage behandeln,social,unternehmen,maßnahme,besuch,geschlachtet tier,aktueller verfahrensstand,staatsministerium
5,lösung,protokoll,geringfügig,social media,menge,einsatz,wieviel,geschlachtet,gemarkung,stand
6,lösung fach,liste,behandeln,erfolgt verstoß,entrichten,stadt,kontrolle,abgeschlossen berichtsjahr,aktueller,berechtigt
7,maßregelvollzug,kommunikation,einschlägig bürgeranfrage,lebensmittelkontrolleur überwachung,kommunal trinkwasser,bereich,bayrischzell,behördlich,bebauungsplan,richtlinie
8,waffenschein,grundwasser,geringfügig gebühr,zuständigkeitsgebiet veröffentlichen,trinkwasser,patient,person,schlachthof,verfolgen,rettungsdienst
9,amtsärztlich,dienst,gebühr fallen,betrieb betrauen,kalenderjahr,gerne,hiermit,dokumentiert,satzungsbeschluß,reaktivierung


## TF-IDF + LSA (DE)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,bußgeld,verfahrensstand,verbraucherinformation,fach,schlachthof,maßregelvollzug,fördermeng,gesamtabrechnunge auflistung,gemarkung flurstücksnr,waffenschein
1,betrieb,bebauungsplanverfahr,kosten,erwartungshorizonte lösung,tier,amtsärztlich begehung,genehmigt,explizit hinweisen,termin planunterlag,verfassungsfeindlich vereinigung
2,durchzuführend risikobeurteilung,baubeginn,umweltinformation,aufgabe erwartungshorizonte,dokumentiert geschlachtet,klinik maßregelvollzug,genehmigt fördermeng,vorliegend dateiformat,aufstellungsbeschluss offenlegung,verfassungsfeindlich
3,durchzuführend,gemeindegebiet,kosten geringfügig,erwartungshorizonte,behördlich überwachung,krankenhaus klinik,unternehmen,gesamtabrechnunge,kurz bebauungsplanverfahr,vereinigung
4,zuständigkeitsgebiet veröffentlichen,aktueller verfahrensstand,geringfügig,aufgabe,geschlachtet tier,bericht gutachten,menge,hinweisen personenbezogen,fortführung verfolgt,afd
5,zuständigkeitsgebiet,satzungsbeschluß,bürgeranfrage behandeln,lösung,geschlachtet,amtsärztlich,kalenderjahr,wirkstoffgehalt,offenlegung bebauungsplanentwurf,waffenbesitzkarte
6,kontrollhäufigkeit erfolgt,offiziell baubeginn,bürgeranfrage,lösung fach,tierart,gesundheitsamt krankenhaus,kommunal trinkwasser,wirkstoffgehalt maßregelvollzug,bebauungsplan link,waffenbesitzkarte waffenschein
7,kontrollierend,bebauungsplanentwurf,behandeln,fach deutsch,abgeschlossen berichtsjahr,begehung gesundheitsamt,trinkwasser,menge wirkstoffgehalt,satzungsbeschluß termin,mitglied
8,kontrollhäufigkeit,aufstellungsbeschluss,einschlägig bürgeranfrage,deutsch,berichtsjahr behördlich,gutachten amtsärztlich,entrichten,maßregelvollzug einkaufen,flurstücksnr,alternative
9,werbeaussag unzulässig,bebauungsplanverfahr projektname,einschlägig,physik,berichtsjahr,durchführen beziehen,wasser,bestellen,aktiv kurz,verfassungsschutz


In [88]:
models_topic_de = {
    "BoW/LDA (de)": {
        "model": lda_bow_topics_by_lang["de"]["model"],
        "features": lda_bow_topics_by_lang["de"]["features"]
    },

    "TF-IDF/LDA (de)": {
        "model": lda_tfidf_topics_by_lang["de"]["model"],
        "features": lda_tfidf_topics_by_lang["de"]["features"]
    },

    "TF-IDF/LSA (de)": {
        "model": lsa_tfidf_topics_by_lang["de"]["model"],
        "features": lsa_tfidf_topics_by_lang["de"]["features"]
    }
}

models_topic_en = {
    "BoW/LDA (en)": {
        "model": lda_bow_topics_by_lang["en"]["model"],
        "features": lda_bow_topics_by_lang["en"]["features"]
    },

    "TF-IDF/LDA (en)": {
        "model": lda_tfidf_topics_by_lang["en"]["model"],
        "features": lda_tfidf_topics_by_lang["en"]["features"]
    },

    "TF-IDF/LSA (en)": {
        "model": lsa_tfidf_topics_by_lang["en"]["model"],
        "features": lsa_tfidf_topics_by_lang["en"]["features"]
    }
}

df_topic_comparison_de = compare_topic_models_multiindex(
    models_topic_de,
    n_top_words
)

df_topic_comparison_en = compare_topic_models_multiindex(
    models_topic_en,
    n_top_words
)

display(df_topic_comparison_de)
display(df_topic_comparison_en)

Topic 1                               \
                 BoW/LDA (de)              TF-IDF/LDA (de)   
0             verfahrensstand                      aufgabe   
1                   baubeginn          erwartungshorizonte   
2        bebauungsplanverfahr                         fach   
3              gemeindegebiet  aufgabe erwartungshorizonte   
4                   gemarkung   erwartungshorizonte lösung   
5                   aktueller                       lösung   
6               bebauungsplan                  lösung fach   
7   aktueller verfahrensstand              maßregelvollzug   
8            satzungsbeschluß                 waffenschein   
9                   verfolgen                 amtsärztlich   
10                      kopie        amtsärztlich begehung   
11                   beginnen       klinik maßregelvollzug   
12                  weiterhin   gesundheitsamt krankenhaus   
13                      aktiv            bericht gutachten   
14                fortführung      begehung gesundheitsamt   

                                                   Topic 2                    \
                         TF-IDF/LSA (de)      BoW/LDA (de)   TF-IDF/LDA (de)   
0                                bußgeld         unterlage         gutachten   
1                                betrieb            schule  klimaneutralität   
2       durchzuführend risikobeurteilung       ministerium         aktivität   
3                         durchzuführend             thema         lehrkraft   
4   zuständigkeitsgebiet veröffentlichen     kommunikation       ministerium   
5                   zuständigkeitsgebiet         politisch         protokoll   
6             kontrollhäufigkeit erfolgt           vertrag             liste   
7                         kontrollierend         aktivität     kommunikation   
8                     kontrollhäufigkeit            nutzen       grundwasser   
9                 werbeaussag unzulässig  klimaneutralität            dienst   
10                     verhängen bußgeld            umwelt            nutzen   
11                              kosmetik             geben            schule   
12                       bußgeld werbung          regelung            bürger   
13          betrauen vollzeitäquivalente        berechtigt         bezüglich   
14            maßgabe kontrollhäufigkeit      organisation    schriftverkehr   

                                                          Topic 3  \
                          TF-IDF/LSA (de)            BoW/LDA (de)   
0                         verfahrensstand  verbraucherinformation   
1                    bebauungsplanverfahr                  kosten   
2                               baubeginn       umweltinformation   
3                          gemeindegebiet                  gebühr   
4               aktueller verfahrensstand             einschlägig   
5                        satzungsbeschluß                  fallen   
6                     offiziell baubeginn             geringfügig   
7                    bebauungsplanentwurf       datenschutzgesetz   
8                   aufstellungsbeschluss                  ablauf   
9        bebauungsplanverfahr projektname                 erbeten   
10                  projektname gemarkung          erbeten ablauf   
11                    baubeginn bauarbeit                     uig   
12  bebauungsplanentwurf satzungsbeschluß      kosten geringfügig   
13                           planunterlag            verbesserung   
14                     planunterlag kopie           gebühr fallen   

                                                                    \
              TF-IDF/LDA (de)                      TF-IDF/LSA (de)   
0      verbraucherinformation               verbraucherinformation   
1                      kosten                               kosten   
2                 einschlägig                    umweltinformation   
3          kosten geringfügig                   kosten geringfügig   
4     bürger

Topic 1                                      Topic 2                  \
   BoW/LDA (en) TF-IDF/LDA (en) TF-IDF/LSA (en) BoW/LDA (en) TF-IDF/LDA (en)   
0         pizza           pizza           order      kitchen            wine   
1          hair           order         service         nail         awesome   
2           use            hair            wait         open         service   
3    absolutely           table      restaurant      service         wedding   
4          beef           drink          burger        owner            beer   
5         close            wait             eat          fan       wonderful   
6         serve         service           drink        later       bartender   
7        flavor      restaurant           price        onion           music   
8        burger         awesome         chicken        drive       beautiful   
9          pork      vegetarian           pizza      finally           close   
10          cut          friend            menu       indian            ring   
11        order             pay           table          hot        attitude   
12         beer             eat             bar          dog            tour   
13          kid           clean           fresh    excellent        employee   
14        style            walk           salad         hope           notch   

                        Topic 3                                      Topic 4  \
   TF-IDF/LSA (en) BoW/LDA (en) TF-IDF/LDA (en) TF-IDF/LSA (en) BoW/LDA (en)   
0           burger         room          burger           pizza        fresh   
1            pizza        hotel           sushi            beer        salad   
2          chicken        check           donut             car        order   
3            salad      awesome          frozen           crust        bread   
4              fry      seating             kid     order pizza         menu   
5            order        treat           treat            thin      service   
6            sauce      wedding         chicken            door         meat   
7           cheese        guest             pet           plain          eat   
8             beer       burger             eat            stay         wine   
9            fresh        happy       excellent        delivery      chicken   
10           lunch          car          family         awesome         fish   
11          flavor         walk            play            cold         meal   
12           serve          big    cheeseburger           dough        taste   
13           taste    beautiful     cheesesteak            room        sauce   
14            meat        small            hard            call      portion   

                                          Topic 5                  \
   TF-IDF/LDA (en)   TF-IDF/LSA (en) BoW/LDA (en) TF-IDF/LDA (en)   
0            salad             sushi        clean            room   
1            order           service         stay           store   
2             taco              roll        large            stay   
3          service             price         door             car   
4          chicken             pizza       flavor           hotel   
5            sauce           chicken         type         helpful   
6            bread             fresh   restaurant           clean   
7             meat  customer service         room            nail   
8              eat               eat      italian       nashville   
9             meal          favorite         star         service   
10            fish         excellent         away            high   
11      restaurant          customer        table           price   
12            rice        reasonable      seafood          person   
13           steak             quick   atmosphere            year   
14          dinner      service slow         dish         parking   

                          Topic 6                                  \
     TF-IDF/LSA (en) BoW/LDA (en) TF-IDF/LD

In [89]:
from IPython.display import display, Markdown

display(Markdown("## TF-IDF + LSA (EN)"))
display(df_tfidf_lsa_en)

display(Markdown("## TF-IDF + LSA (DE)"))
display(df_tfidf_lsa_de)

## TF-IDF + LSA (EN)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,order,burger,pizza,sushi,burger,order,sushi,beer,room,breakfast
1,service,pizza,beer,service,service,drink,room,service,hotel,service
2,wait,chicken,car,roll,car,bar,roll,wine,excellent,coffee
3,restaurant,salad,crust,price,sushi,table,burger,bar,service,awesome
4,burger,fry,order pizza,pizza,fry,restaurant,hotel,awesome,chicken,sandwich
5,eat,order,thin,chicken,customer,night,pizza,selection,stay,excellent
6,drink,sauce,door,fresh,customer service,beer,clean,price,sauce,wait
7,price,cheese,plain,customer service,order,sit,stay,drink,dinner,pancake
8,chicken,beer,stay,eat,excellent,server,beer,store,dish,breakfast sandwich
9,pizza,fresh,delivery,favorite,slow,sushi,breakfast,atmosphere,awesome,line


## TF-IDF + LSA (DE)

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,bußgeld,verfahrensstand,verbraucherinformation,fach,schlachthof,maßregelvollzug,fördermeng,gesamtabrechnunge auflistung,gemarkung flurstücksnr,waffenschein
1,betrieb,bebauungsplanverfahr,kosten,erwartungshorizonte lösung,tier,amtsärztlich begehung,genehmigt,explizit hinweisen,termin planunterlag,verfassungsfeindlich vereinigung
2,durchzuführend risikobeurteilung,baubeginn,umweltinformation,aufgabe erwartungshorizonte,dokumentiert geschlachtet,klinik maßregelvollzug,genehmigt fördermeng,vorliegend dateiformat,aufstellungsbeschluss offenlegung,verfassungsfeindlich
3,durchzuführend,gemeindegebiet,kosten geringfügig,erwartungshorizonte,behördlich überwachung,krankenhaus klinik,unternehmen,gesamtabrechnunge,kurz bebauungsplanverfahr,vereinigung
4,zuständigkeitsgebiet veröffentlichen,aktueller verfahrensstand,geringfügig,aufgabe,geschlachtet tier,bericht gutachten,menge,hinweisen personenbezogen,fortführung verfolgt,afd
5,zuständigkeitsgebiet,satzungsbeschluß,bürgeranfrage behandeln,lösung,geschlachtet,amtsärztlich,kalenderjahr,wirkstoffgehalt,offenlegung bebauungsplanentwurf,waffenbesitzkarte
6,kontrollhäufigkeit erfolgt,offiziell baubeginn,bürgeranfrage,lösung fach,tierart,gesundheitsamt krankenhaus,kommunal trinkwasser,wirkstoffgehalt maßregelvollzug,bebauungsplan link,waffenbesitzkarte waffenschein
7,kontrollierend,bebauungsplanentwurf,behandeln,fach deutsch,abgeschlossen berichtsjahr,begehung gesundheitsamt,trinkwasser,menge wirkstoffgehalt,satzungsbeschluß termin,mitglied
8,kontrollhäufigkeit,aufstellungsbeschluss,einschlägig bürgeranfrage,deutsch,berichtsjahr behördlich,gutachten amtsärztlich,entrichten,maßregelvollzug einkaufen,flurstücksnr,alternative
9,werbeaussag unzulässig,bebauungsplanverfahr projektname,einschlägig,physik,berichtsjahr,durchführen beziehen,wasser,bestellen,aktiv kurz,verfassungsschutz


In [95]:
def flatten_columns(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" | ".join(map(str, col)).strip() for col in df.columns]
    return df

def highlight_duplicates(df):
    df = flatten_columns(df).astype(str).replace({"nan": "", "None": ""})

    counts = pd.Series(df.to_numpy().ravel())
    counts = counts[counts != ""].value_counts()

    def color_cell(value):
        freq = counts.get(str(value), 0)
        if freq >= 5:
            return "background-color: #f4b183"
        elif freq >= 3:
            return "background-color: #ffd966"
        elif freq >= 2:
            return "background-color: #fff2cc"
        return ""

    return df.style.map(color_cell)

def show_topic_tables(tables, language):
    display(Markdown(f"# Topic-Vergleich {language}"))

    for title, df in tables.items():
        display(Markdown(f"## {title}"))
        display(highlight_duplicates(df))

tables_de = {
    "BoW + LDA": df_bow_lda_de,
    "TF-IDF + LDA": df_tfidf_lda_de,
    "TF-IDF + LSA": df_tfidf_lsa_de
}

tables_en = {
    "BoW + LDA": df_bow_lda_en,
    "TF-IDF + LDA": df_tfidf_lda_en,
    "TF-IDF + LSA": df_tfidf_lsa_en
}

show_topic_tables(tables_de, "Deutsch")
show_topic_tables(tables_en, "Englisch")

# Topic-Vergleich Deutsch

## BoW + LDA

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,verfahrensstand,unterlage,verbraucherinformation,betrieb,maßnahme,tier,münchen,arzt,waffenschein,fördermeng
1,baubeginn,schule,kosten,bußgeld,vollständig,behördlich,straße,medizinisch,verfassungsfeindlich,unternehmen
2,bebauungsplanverfahr,ministerium,umweltinformation,werbung,biber,bundesland,stadt,meldung,afd,genehmigt
3,gemeindegebiet,thema,gebühr,plattform,konkret,aufschlüsseln,person,unerwünscht,vereinigung,genehmigt fördermeng
4,gemarkung,kommunikation,einschlägig,unzulässig,gebiet,dokumentiert,finden,verdacht,verfassungsfeindlich vereinigung,kommunal
5,aktueller,politisch,fallen,verstoß,staatsministerium,haus,stehen,bevölkerung,person,wasser
6,bebauungsplan,vertrag,geringfügig,erfolgt,regierung,überwachung,derzeit,unterweisung,mitglied,genehmigung
7,aktueller verfahrensstand,aktivität,datenschutzgesetz,herstellen,zeitraum,schlachthof,frau,deutsch,waffenbesitzkarte,menge
8,satzungsbeschluß,nutzen,ablauf,berücksichtigung,artikel,differenzieren,laut,hintergrund,waffenbesitzkarte waffenschein,entrichten
9,verfolgen,klimaneutralität,erbeten,antworten,genehmigung,betrieb,maßnahme,verfügung,alternative,kalenderjahr


## TF-IDF + LDA

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,aufgabe,gutachten,verbraucherinformation,bußgeld,fördermeng,münchen,nürnberg,dokumentiert geschlachtet,verfahrensstand,gymnasium
1,erwartungshorizonte,klimaneutralität,kosten,betrieb,genehmigt,polizei,jva,behördlich überwachung,bebauungsplanverfahr,steuer
2,fach,aktivität,einschlägig,werbung,genehmigt fördermeng,straße,frau,berichtsjahr,baubeginn,weisung
3,aufgabe erwartungshorizonte,lehrkraft,kosten geringfügig,werbung social,kommunal,unterlage,umgang,tierart,gemeindegebiet,bay
4,erwartungshorizonte lösung,ministerium,bürgeranfrage behandeln,social,unternehmen,maßnahme,besuch,geschlachtet tier,aktueller verfahrensstand,staatsministerium
5,lösung,protokoll,geringfügig,social media,menge,einsatz,wieviel,geschlachtet,gemarkung,stand
6,lösung fach,liste,behandeln,erfolgt verstoß,entrichten,stadt,kontrolle,abgeschlossen berichtsjahr,aktueller,berechtigt
7,maßregelvollzug,kommunikation,einschlägig bürgeranfrage,lebensmittelkontrolleur überwachung,kommunal trinkwasser,bereich,bayrischzell,behördlich,bebauungsplan,richtlinie
8,waffenschein,grundwasser,geringfügig gebühr,zuständigkeitsgebiet veröffentlichen,trinkwasser,patient,person,schlachthof,verfolgen,rettungsdienst
9,amtsärztlich,dienst,gebühr fallen,betrieb betrauen,kalenderjahr,gerne,hiermit,dokumentiert,satzungsbeschluß,reaktivierung


...

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,pizza,kitchen,room,fresh,clean,order,lunch,price,order,coffee
1,hair,nail,hotel,salad,stay,wait,chicken,service,table,car
2,use,open,check,order,large,taco,drink,store,friend,customer
3,absolutely,service,awesome,bread,door,burger,beer,sushi,night,service
4,beef,owner,seating,menu,flavor,drink,bar,roll,bar,wait
5,close,fan,treat,service,type,bbq,hot,month,restaurant,call
6,serve,later,wedding,meat,restaurant,bean,sandwich,reasonable,sit,pay
7,flavor,onion,guest,eat,room,atmosphere,salad,buy,cake,line
8,burger,drive,burger,wine,italian,tip,tasty,selection,service,bring
9,pork,finally,happy,chicken,star,eat,fry,sandwich,server,let


## TF-IDF + LDA

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,pizza,wine,burger,salad,room,pizza,roll,chicken,cake,wait
1,order,awesome,sushi,order,store,fresh,sushi,ice,change,bar
2,hair,service,donut,taco,stay,fry,tea,ice cream,price,service
3,table,wedding,frozen,service,car,best,soup,cream,service,burger
4,drink,beer,kid,chicken,hotel,burger,lunch,coffee,fair,menu
5,wait,wonderful,treat,sauce,helpful,line,chinese,reasonable,server,car
6,service,bartender,chicken,bread,clean,flavor,oyster,sandwich,rude,coffee
7,restaurant,music,pet,meat,nail,breakfast,cafe,cute,indian,call
8,awesome,beautiful,eat,eat,nashville,perfect,order,chocolate,relaxed,drink
9,vegetarian,close,excellent,meal,service,wall,pho,awesome,attentive,stop


## TF-IDF + LSA

,Topic1,Topic2,Topic3,Topic4,Topic5,Topic6,Topic7,Topic8,Topic9,Topic10
0,order,burger,pizza,sushi,burger,order,sushi,beer,room,breakfast
1,service,pizza,beer,service,service,drink,room,service,hotel,service
2,wait,chicken,car,roll,car,bar,roll,wine,excellent,coffee
3,restaurant,salad,crust,price,sushi,table,burger,bar,service,awesome
4,burger,fry,order pizza,pizza,fry,restaurant,hotel,awesome,chicken,sandwich
5,eat,order,thin,chicken,customer,night,pizza,selection,stay,excellent
6,drink,sauce,door,fresh,customer service,beer,clean,price,sauce,wait
7,price,cheese,plain,customer service,order,sit,stay,drink,dinner,pancake
8,chicken,beer,stay,eat,excellent,server,beer,store,dish,breakfast sandwich
9,pizza,fresh,delivery,favorite,slow,sushi,breakfast,atmosphere,awesome,line
